# Modeling

## Data Input

In [1]:
import pandas as pd
import numpy as ny

In [2]:
df = pd.read_csv("df.csv")

In [3]:
df_cleaned = pd.read_csv("df_cleaned.csv")

## Total Data Frame

### Collinearity

In [4]:
df_encoded = pd.get_dummies(df_cleaned, drop_first=True)

In [5]:
df_num = df.select_dtypes(include=['int64', 'float64']).copy()
print(df_num.isna().sum())

id                                0
day_in_study                      0
lh                                0
estrogen                          0
calories_daily                    0
steps_daily                       0
bpm_mean                          0
bpm_min                           0
bpm_max                           0
bpm_std                           0
temp_diff_from_baseline_mean      0
temp_diff_from_baseline_min       0
temp_diff_from_baseline_max       0
temp_diff_from_baseline_std       0
glucose_mean                      0
glucose_min                       0
glucose_max                       0
glucose_std                       0
overall_score                     0
composition_score                 0
revitalization_score              0
duration_score                    0
nightly_temperature               0
duration                          0
minutestofallasleep               0
minutesasleep                     0
minutesawake                      0
minutesafterwakeup          

In [6]:
df_num = df_num.fillna(df_num.mean())

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_num = df_num.assign(constant=1)

vif_data = pd.DataFrame()
vif_data["feature"] = df_num.columns
vif_data["VIF"] = [
    variance_inflation_factor(df_num.values, i)
    for i in range(df_num.shape[1])
]

vif_data = vif_data[vif_data["feature"] != "constant"]
print(vif_data)


/opt/conda/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


                         feature         VIF
0                             id    1.582630
1                   day_in_study    1.146379
2                             lh    1.145631
3                       estrogen    1.254918
4                 calories_daily    3.534922
5                    steps_daily    2.811288
6                       bpm_mean    8.083730
7                        bpm_min    5.986826
8                        bpm_max    2.448555
9                        bpm_std    4.364768
10  temp_diff_from_baseline_mean    4.174768
11   temp_diff_from_baseline_min    1.952554
12   temp_diff_from_baseline_max    1.927867
13   temp_diff_from_baseline_std    4.459121
14                  glucose_mean  221.044719
15                   glucose_min   85.152916
16                   glucose_max  200.750005
17                   glucose_std   47.894694
18                 overall_score         inf
19             composition_score         inf
20          revitalization_score         inf
21        

In [8]:
drop_cols = [
    "composition_score",
    "revitalization_score",
    "duration_score",
    "glucose_mean",
    "glucose_min",
    "glucose_std",
    "minutesasleep",
    "duration_exercise",
    "calories_exercise"
]

In [9]:
df_clean = df.drop(columns=drop_cols)

### df_clean Modeling

#### Preprocessing

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

TARGET_COL = "phase"

X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

mask = y.notna()
X = X[mask]
y = y[mask]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Categorical cols: ['flow_volume', 'flow_color', 'appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion', 'bloating', 'activity_type', 'gender', 'ethnicity', 'education', 'sexually_active', 'self_report_menstrual_health_literacy']
Numeric cols: ['id', 'day_in_study', 'lh', 'estrogen', 'calories_daily', 'steps_daily', 'bpm_mean', 'bpm_min', 'bpm_max', 'bpm_std', 'temp_diff_from_baseline_mean', 'temp_diff_from_baseline_min', 'temp_diff_from_baseline_max', 'temp_diff_from_baseline_std', 'glucose_max', 'overall_score', 'nightly_temperature', 'duration', 'minutestofallasleep', 'minutesawake', 'minutesafterwakeup', 'timeinbed', 'efficiency', 'heartrate_exercise', 'in_default_zone_3', 'in_default_zone_2', 'in_default_zone_1', 'below_default_zone_1', 'birth_year', 'age_of_first_menarche', 'flow_volume_score', 'appetite_score', 'exerciselevel_score', 'headaches_score', 'cramps_score', 'sorebreasts_score', 'f

#### Decision Tree

In [11]:
from sklearn.tree import DecisionTreeClassifier

dt_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))



===== Decision Tree =====
Accuracy: 0.5420875420875421
Macro F1: 0.5423005797892367
Macro AUC: 0.6916423639756734
              precision    recall  f1-score   support

   Fertility       0.42      0.44      0.43       131
  Follicular       0.51      0.55      0.53       161
      Luteal       0.60      0.58      0.59       191
   Menstrual       0.66      0.59      0.62       111

    accuracy                           0.54       594
   macro avg       0.55      0.54      0.54       594
weighted avg       0.55      0.54      0.54       594



#### Random Forest

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))




===== Random Forest =====
Accuracy: 0.6818181818181818
Macro F1: 0.669505239190374
Macro AUC: 0.8830113017861096
              precision    recall  f1-score   support

   Fertility       0.72      0.47      0.56       131
  Follicular       0.72      0.65      0.68       161
      Luteal       0.62      0.87      0.73       191
   Menstrual       0.75      0.67      0.70       111

    accuracy                           0.68       594
   macro avg       0.70      0.66      0.67       594
weighted avg       0.69      0.68      0.67       594



#### Logistics Regression

In [13]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        solver="lbfgs",
        class_weight="balanced",
        n_jobs=-1
    ))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_proba = log_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))



/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



===== Logistic Regression =====
Accuracy: 0.5959595959595959
Macro F1: 0.5965258692680263
Macro AUC: 0.832703184777051
              precision    recall  f1-score   support

   Fertility       0.48      0.46      0.47       131
  Follicular       0.55      0.61      0.58       161
      Luteal       0.65      0.62      0.64       191
   Menstrual       0.72      0.68      0.70       111

    accuracy                           0.60       594
   macro avg       0.60      0.60      0.60       594
weighted avg       0.60      0.60      0.60       594



#### XGBoost

In [14]:
from sklearn.preprocessing import LabelEncoder

y_raw = df_clean[TARGET_COL]

le = LabelEncoder()
y = le.fit_transform(y_raw)

mask = ~pd.isna(y)
X = df_clean.drop(columns=[TARGET_COL])[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [15]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

# class weight（balanced）
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))

sample_weight = np.array([class_weight_dict[c] for c in y_train])

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train, clf__sample_weight=sample_weight)

y_pred_int = xgb_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_int)

y_proba = xgb_model.predict_proba(X_test)

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
auc_macro = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

print("\n===== XGBoost =====")
print("Accuracy:", accuracy_score(le.inverse_transform(y_test), y_pred))
print("Macro F1:", f1_score(le.inverse_transform(y_test), y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(le.inverse_transform(y_test), y_pred))



===== XGBoost =====
Accuracy: 0.6936026936026936
Macro F1: 0.6896032523691331
Macro AUC: 0.8905532440724828
              precision    recall  f1-score   support

   Fertility       0.61      0.59      0.60       131
  Follicular       0.70      0.71      0.71       161
      Luteal       0.70      0.76      0.73       191
   Menstrual       0.80      0.67      0.73       111

    accuracy                           0.69       594
   macro avg       0.70      0.68      0.69       594
weighted avg       0.70      0.69      0.69       594



## ANOVA Chosen Dataframe

### Collinearity

In [16]:
df_anova = df[['flow_volume_score', 'cramps_score', 'lh', 'estrogen', 'flow_color', 'sorebreasts_score', 'nightly_temperature', 
               'bpm_min', 'bpm_mean', 'bloating_score', 'temp_diff_from_baseline_max', 'revitalization_score', 
               'in_default_zone_1', 'indigestion_score', 'headaches_score','phase']]

In [17]:
df_anova_num = df_anova.select_dtypes(include=['int64', 'float64']).copy()
print(df_anova_num.isna().sum())

flow_volume_score              392
cramps_score                     0
lh                               0
estrogen                         0
sorebreasts_score                0
nightly_temperature              0
bpm_min                          0
bpm_mean                         0
bloating_score                   0
temp_diff_from_baseline_max      0
revitalization_score             0
in_default_zone_1                0
indigestion_score                0
headaches_score                  0
dtype: int64


In [18]:
df_anova_num = df_anova_num.fillna(df_anova_num.mean())

In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_anova_num = df_anova_num.assign(constant=1)

vif_data_anova = pd.DataFrame()
vif_data_anova["feature"] = df_anova_num.columns
vif_data_anova["VIF"] = [
    variance_inflation_factor(df_anova_num.values, i)
    for i in range(df_anova_num.shape[1])
]

vif_data_anova = vif_data_anova[vif_data_anova["feature"] != "constant"]
print(vif_data_anova)

                        feature       VIF
0             flow_volume_score  1.098447
1                  cramps_score  1.687312
2                            lh  1.097116
3                      estrogen  1.123030
4             sorebreasts_score  1.609997
5           nightly_temperature  1.102358
6                       bpm_min  2.900279
7                      bpm_mean  4.698840
8                bloating_score  2.086788
9   temp_diff_from_baseline_max  1.121224
10         revitalization_score  1.096661
11            in_default_zone_1  2.233027
12            indigestion_score  2.015502
13              headaches_score  1.218477


### df_anova Modeling

#### Preprocessing

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

TARGET_COL = "phase"

X = df_anova.drop(columns=[TARGET_COL])
y = df_anova[TARGET_COL]

mask = y.notna()
X = X[mask]
y = y[mask]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Categorical cols: ['flow_color']
Numeric cols: ['flow_volume_score', 'cramps_score', 'lh', 'estrogen', 'sorebreasts_score', 'nightly_temperature', 'bpm_min', 'bpm_mean', 'bloating_score', 'temp_diff_from_baseline_max', 'revitalization_score', 'in_default_zone_1', 'indigestion_score', 'headaches_score']


#### Decision Tree

In [21]:
from sklearn.tree import DecisionTreeClassifier

dt_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))



===== Decision Tree =====
Accuracy: 0.5235690235690236
Macro F1: 0.524814004959241
Macro AUC: 0.6831844309766785
              precision    recall  f1-score   support

   Fertility       0.43      0.45      0.44       131
  Follicular       0.43      0.44      0.44       161
      Luteal       0.61      0.59      0.60       191
   Menstrual       0.63      0.61      0.62       111

    accuracy                           0.52       594
   macro avg       0.53      0.52      0.52       594
weighted avg       0.53      0.52      0.53       594



#### Random Forest

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))



===== Random Forest =====
Accuracy: 0.6464646464646465
Macro F1: 0.6393535982692033
Macro AUC: 0.8563007855800543
              precision    recall  f1-score   support

   Fertility       0.70      0.47      0.56       131
  Follicular       0.59      0.65      0.62       161
      Luteal       0.64      0.77      0.70       191
   Menstrual       0.73      0.65      0.69       111

    accuracy                           0.65       594
   macro avg       0.66      0.63      0.64       594
weighted avg       0.65      0.65      0.64       594



#### Logistic Regression

In [23]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        solver="lbfgs",
        class_weight="balanced",
        n_jobs=-1
    ))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_proba = log_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



===== Logistic Regression =====
Accuracy: 0.569023569023569
Macro F1: 0.565458125283879
Macro AUC: 0.8118151181288732
              precision    recall  f1-score   support

   Fertility       0.50      0.41      0.45       131
  Follicular       0.49      0.50      0.50       161
      Luteal       0.64      0.64      0.64       191
   Menstrual       0.63      0.73      0.68       111

    accuracy                           0.57       594
   macro avg       0.56      0.57      0.57       594
weighted avg       0.56      0.57      0.57       594



#### XGBoost

In [24]:
from sklearn.preprocessing import LabelEncoder

y_raw = df_anova[TARGET_COL]

le = LabelEncoder()
y = le.fit_transform(y_raw)

mask = ~pd.isna(y)
X = df_anova.drop(columns=[TARGET_COL])[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

# class weight（balanced）
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))

sample_weight = np.array([class_weight_dict[c] for c in y_train])

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train, clf__sample_weight=sample_weight)

y_pred_int = xgb_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_int)

y_proba = xgb_model.predict_proba(X_test)

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
auc_macro = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

print("\n===== XGBoost =====")
print("Accuracy:", accuracy_score(le.inverse_transform(y_test), y_pred))
print("Macro F1:", f1_score(le.inverse_transform(y_test), y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(le.inverse_transform(y_test), y_pred))


===== XGBoost =====
Accuracy: 0.6195286195286195
Macro F1: 0.6195210001018634
Macro AUC: 0.8405998851841854
              precision    recall  f1-score   support

   Fertility       0.54      0.51      0.53       131
  Follicular       0.55      0.60      0.57       161
      Luteal       0.68      0.68      0.68       191
   Menstrual       0.72      0.68      0.70       111

    accuracy                           0.62       594
   macro avg       0.62      0.62      0.62       594
weighted avg       0.62      0.62      0.62       594



## K-Best

### Collinearity

In [26]:
df_k = df[['lh', 'estrogen' ,'bpm_min', 'nightly_temperature',
 'flow_volume_score', 'cramps_score', 'sorebreasts_score',
 'flow_color','phase']]

In [27]:
df_k_num = df_k.select_dtypes(include=['int64', 'float64']).copy()
print(df_k_num.isna().sum())

lh                       0
estrogen                 0
bpm_min                  0
nightly_temperature      0
flow_volume_score      392
cramps_score             0
sorebreasts_score        0
dtype: int64


In [28]:
df_k_num = df_k_num.fillna(df_k_num.mean())

In [29]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_k_num = df_k_num.assign(constant=1)

vif_data_k = pd.DataFrame()
vif_data_k["feature"] = df_k_num.columns
vif_data_k["VIF"] = [
    variance_inflation_factor(df_k_num.values, i)
    for i in range(df_k_num.shape[1])
]

vif_data_k = vif_data_k[vif_data_k["feature"] != "constant"]
print(vif_data_k)

               feature       VIF
0                   lh  1.082375
1             estrogen  1.109185
2              bpm_min  1.064253
3  nightly_temperature  1.009040
4    flow_volume_score  1.085991
5         cramps_score  1.569890
6    sorebreasts_score  1.518490


### df_k Modeling

#### Preprocessing

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

TARGET_COL = "phase"

X = df_k.drop(columns=[TARGET_COL])
y = df_k[TARGET_COL]

mask = y.notna()
X = X[mask]
y = y[mask]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Categorical cols: ['flow_color']
Numeric cols: ['lh', 'estrogen', 'bpm_min', 'nightly_temperature', 'flow_volume_score', 'cramps_score', 'sorebreasts_score']


#### Decision Tree

In [31]:
from sklearn.tree import DecisionTreeClassifier

dt_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))


===== Decision Tree =====
Accuracy: 0.5117845117845118
Macro F1: 0.512913191061086
Macro AUC: 0.6727934614474742
              precision    recall  f1-score   support

   Fertility       0.45      0.37      0.41       131
  Follicular       0.38      0.40      0.39       161
      Luteal       0.57      0.63      0.60       191
   Menstrual       0.68      0.63      0.65       111

    accuracy                           0.51       594
   macro avg       0.52      0.51      0.51       594
weighted avg       0.51      0.51      0.51       594



#### Random Forest

In [32]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))



===== Random Forest =====
Accuracy: 0.5875420875420876
Macro F1: 0.5806748734471383
Macro AUC: 0.8238994811178726
              precision    recall  f1-score   support

   Fertility       0.57      0.41      0.48       131
  Follicular       0.51      0.54      0.52       161
      Luteal       0.61      0.72      0.66       191
   Menstrual       0.68      0.64      0.66       111

    accuracy                           0.59       594
   macro avg       0.59      0.58      0.58       594
weighted avg       0.59      0.59      0.58       594



#### Logistic Regression

In [33]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        solver="lbfgs",
        class_weight="balanced",
        n_jobs=-1
    ))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_proba = log_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



===== Logistic Regression =====
Accuracy: 0.5723905723905723
Macro F1: 0.5708993626712694
Macro AUC: 0.808768652948011
              precision    recall  f1-score   support

   Fertility       0.50      0.43      0.46       131
  Follicular       0.52      0.51      0.51       161
      Luteal       0.60      0.63      0.62       191
   Menstrual       0.65      0.73      0.69       111

    accuracy                           0.57       594
   macro avg       0.57      0.58      0.57       594
weighted avg       0.57      0.57      0.57       594



#### XGBoost

In [34]:
from sklearn.preprocessing import LabelEncoder

y_raw = df_k[TARGET_COL]

le = LabelEncoder()
y = le.fit_transform(y_raw)

mask = ~pd.isna(y)
X = df_k.drop(columns=[TARGET_COL])[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

# class weight（balanced）
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))

sample_weight = np.array([class_weight_dict[c] for c in y_train])

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train, clf__sample_weight=sample_weight)

y_pred_int = xgb_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_int)

y_proba = xgb_model.predict_proba(X_test)

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
auc_macro = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

print("\n===== XGBoost =====")
print("Accuracy:", accuracy_score(le.inverse_transform(y_test), y_pred))
print("Macro F1:", f1_score(le.inverse_transform(y_test), y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(le.inverse_transform(y_test), y_pred))


===== XGBoost =====
Accuracy: 0.6043771043771043
Macro F1: 0.6062404512302424
Macro AUC: 0.8290850165231708
              precision    recall  f1-score   support

   Fertility       0.55      0.48      0.51       131
  Follicular       0.51      0.55      0.53       161
      Luteal       0.66      0.65      0.66       191
   Menstrual       0.71      0.75      0.73       111

    accuracy                           0.60       594
   macro avg       0.61      0.61      0.61       594
weighted avg       0.60      0.60      0.60       594



## Forward Selection Dataframe

### Collinearity

In [78]:
df_forward = df[['lh', 'estrogen', 'calories_daily', 'temp_diff_from_baseline_max', 'glucose_mean', 'glucose_std', 'revitalization_score', 'minutestofallasleep', 'minutesasleep', 'minutesafterwakeup', 'duration_exercise', 'calories_exercise', 'heartrate_exercise', 'in_default_zone_3', 'in_default_zone_2', 'flow_volume_score', 'appetite_score', 'cramps_score', 'sorebreasts_score', 'fatigue_score', 'sleepissue_score', 'foodcravings_score', 'flow_color', 'activity_type', 'gender', 'ethnicity', 'education', 'sexually_active', 'self_report_menstrual_health_literacy', 'phase']]

In [79]:
df_forward_num = df_forward.select_dtypes(include=['int64', 'float64']).copy()
print(df_forward_num.isna().sum())

lh                               0
estrogen                         0
calories_daily                   0
temp_diff_from_baseline_max      0
glucose_mean                     0
glucose_std                      0
revitalization_score             0
minutestofallasleep              0
minutesasleep                    0
minutesafterwakeup               0
duration_exercise                0
calories_exercise                0
heartrate_exercise               0
in_default_zone_3                0
in_default_zone_2                0
flow_volume_score              392
appetite_score                   0
cramps_score                     0
sorebreasts_score                0
fatigue_score                    0
sleepissue_score                 0
foodcravings_score               0
dtype: int64


In [80]:
df_forward_num = df_forward_num.fillna(df_forward_num.mean())

In [81]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_forward_num = df_forward_num.assign(constant=1)

vif_data_forward = pd.DataFrame()
vif_data_forward["feature"] = df_forward_num.columns
vif_data_forward["VIF"] = [
    variance_inflation_factor(df_forward_num.values, i)
    for i in range(df_forward_num.shape[1])
]

vif_data_forward = vif_data_forward[vif_data_forward["feature"] != "constant"]
print(vif_data_forward)

                        feature        VIF
0                            lh   1.108486
1                      estrogen   1.136106
2                calories_daily   1.409880
3   temp_diff_from_baseline_max   1.112638
4                  glucose_mean   8.018807
5                   glucose_std   7.832530
6          revitalization_score   1.085680
7           minutestofallasleep   1.032751
8                 minutesasleep   1.077320
9            minutesafterwakeup   1.048570
10            duration_exercise  19.542257
11            calories_exercise  21.203995
12           heartrate_exercise   2.752116
13            in_default_zone_3   1.148792
14            in_default_zone_2   1.338203
15            flow_volume_score   1.097424
16               appetite_score   1.152147
17                 cramps_score   1.632342
18            sorebreasts_score   1.585794
19                fatigue_score   1.404357
20             sleepissue_score   1.463341
21           foodcravings_score   1.402029


### df_forward Modeling

#### Preprocessing

In [82]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

TARGET_COL = "phase"

X = df_forward.drop(columns=[TARGET_COL])
y = df_forward[TARGET_COL]

mask = y.notna()
X = X[mask]
y = y[mask]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Categorical cols: ['flow_color', 'activity_type', 'gender', 'ethnicity', 'education', 'sexually_active', 'self_report_menstrual_health_literacy']
Numeric cols: ['lh', 'estrogen', 'calories_daily', 'temp_diff_from_baseline_max', 'glucose_mean', 'glucose_std', 'revitalization_score', 'minutestofallasleep', 'minutesasleep', 'minutesafterwakeup', 'duration_exercise', 'calories_exercise', 'heartrate_exercise', 'in_default_zone_3', 'in_default_zone_2', 'flow_volume_score', 'appetite_score', 'cramps_score', 'sorebreasts_score', 'fatigue_score', 'sleepissue_score', 'foodcravings_score']


#### Decision Tree

In [83]:
from sklearn.tree import DecisionTreeClassifier

dt_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))


===== Decision Tree =====
Accuracy: 0.48148148148148145
Macro F1: 0.48333058767020354
Macro AUC: 0.6506322476974746
              precision    recall  f1-score   support

   Fertility       0.42      0.40      0.41       131
  Follicular       0.40      0.42      0.41       161
      Luteal       0.54      0.54      0.54       191
   Menstrual       0.59      0.57      0.58       111

    accuracy                           0.48       594
   macro avg       0.49      0.48      0.48       594
weighted avg       0.48      0.48      0.48       594



#### Random Forest

In [84]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))


===== Random Forest =====
Accuracy: 0.6599326599326599
Macro F1: 0.6447824860900848
Macro AUC: 0.869659660320803
              precision    recall  f1-score   support

   Fertility       0.68      0.45      0.54       131
  Follicular       0.67      0.58      0.62       161
      Luteal       0.62      0.88      0.73       191
   Menstrual       0.73      0.65      0.69       111

    accuracy                           0.66       594
   macro avg       0.68      0.64      0.64       594
weighted avg       0.67      0.66      0.65       594



#### Logistic Regression

In [85]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        solver="lbfgs",
        class_weight="balanced",
        n_jobs=-1
    ))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_proba = log_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



===== Logistic Regression =====
Accuracy: 0.5707070707070707
Macro F1: 0.5729894430007794
Macro AUC: 0.8163871897503254
              precision    recall  f1-score   support

   Fertility       0.50      0.44      0.47       131
  Follicular       0.51      0.53      0.52       161
      Luteal       0.60      0.61      0.61       191
   Menstrual       0.68      0.72      0.70       111

    accuracy                           0.57       594
   macro avg       0.57      0.57      0.57       594
weighted avg       0.57      0.57      0.57       594



#### XGBoost

In [86]:
from sklearn.preprocessing import LabelEncoder

y_raw = df_forward[TARGET_COL]

le = LabelEncoder()
y = le.fit_transform(y_raw)

mask = ~pd.isna(y)
X = df_forward.drop(columns=[TARGET_COL])[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [87]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

# class weight（balanced）
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))

sample_weight = np.array([class_weight_dict[c] for c in y_train])

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train, clf__sample_weight=sample_weight)

y_pred_int = xgb_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_int)

y_proba = xgb_model.predict_proba(X_test)

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
auc_macro = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

print("\n===== XGBoost =====")
print("Accuracy:", accuracy_score(le.inverse_transform(y_test), y_pred))
print("Macro F1:", f1_score(le.inverse_transform(y_test), y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(le.inverse_transform(y_test), y_pred))


===== XGBoost =====
Accuracy: 0.6531986531986532
Macro F1: 0.6481744868457402
Macro AUC: 0.8731810116784884
              precision    recall  f1-score   support

   Fertility       0.61      0.52      0.56       131
  Follicular       0.59      0.58      0.58       161
      Luteal       0.67      0.79      0.72       191
   Menstrual       0.75      0.69      0.72       111

    accuracy                           0.65       594
   macro avg       0.66      0.64      0.65       594
weighted avg       0.65      0.65      0.65       594



## Backward Selection Dataframe

### Collinearity

In [62]:
df_backward = df[['lh','estrogen','bpm_mean','bpm_min',
 'temp_diff_from_baseline_max', 'nightly_temperature',
 'efficiency', 'in_default_zone_2', 'below_default_zone_1',
 'age_of_first_menarche', 'flow_volume_score',
 'exerciselevel_score', 'cramps_score', 'sorebreasts_score',
 'flow_color',"phase"]]

In [63]:
df_backward_num = df_backward.select_dtypes(include=['int64', 'float64']).copy()
print(df_backward_num.isna().sum())

lh                               0
estrogen                         0
bpm_mean                         0
bpm_min                          0
temp_diff_from_baseline_max      0
nightly_temperature              0
efficiency                       0
in_default_zone_2                0
below_default_zone_1             0
age_of_first_menarche            0
flow_volume_score              392
exerciselevel_score              0
cramps_score                     0
sorebreasts_score                0
dtype: int64


In [64]:
df_backward_num = df_backward_num.fillna(df_backward_num.mean())

In [65]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_backward_num = df_backward_num.assign(constant=1)

vif_data_backward = pd.DataFrame()
vif_data_backward["feature"] = df_backward_num.columns
vif_data_backward["VIF"] = [
    variance_inflation_factor(df_backward_num.values, i)
    for i in range(df_backward_num.shape[1])
]

vif_data_backward = vif_data_forward[vif_data_backward["feature"] != "constant"]
print(vif_data_backward)

             feature       VIF
0                 lh  1.000677
1  flow_volume_score  1.000677


/var/tmp/ipykernel_126107/1073549274.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  vif_data_backward = vif_data_forward[vif_data_backward["feature"] != "constant"]


### df_backward Modeling

#### Preprocessing

In [66]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer

TARGET_COL = "phase"

X = df_backward.drop(columns=[TARGET_COL])
y = df_backward[TARGET_COL]

mask = y.notna()
X = X[mask]
y = y[mask]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Categorical cols: ['flow_color']
Numeric cols: ['lh', 'estrogen', 'bpm_mean', 'bpm_min', 'temp_diff_from_baseline_max', 'nightly_temperature', 'efficiency', 'in_default_zone_2', 'below_default_zone_1', 'age_of_first_menarche', 'flow_volume_score', 'exerciselevel_score', 'cramps_score', 'sorebreasts_score']


#### Decision Tree

In [67]:
from sklearn.tree import DecisionTreeClassifier

dt_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_proba = dt_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))


===== Decision Tree =====
Accuracy: 0.5218855218855218
Macro F1: 0.5207868253741724
Macro AUC: 0.6813128461703555
              precision    recall  f1-score   support

   Fertility       0.45      0.46      0.45       131
  Follicular       0.48      0.42      0.45       161
      Luteal       0.56      0.60      0.58       191
   Menstrual       0.60      0.60      0.60       111

    accuracy                           0.52       594
   macro avg       0.52      0.52      0.52       594
weighted avg       0.52      0.52      0.52       594



#### Random Forest

In [68]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))


===== Random Forest =====
Accuracy: 0.6481481481481481
Macro F1: 0.6358892447390393
Macro AUC: 0.8573522885452819
              precision    recall  f1-score   support

   Fertility       0.70      0.43      0.53       131
  Follicular       0.59      0.66      0.62       161
      Luteal       0.66      0.78      0.71       191
   Menstrual       0.70      0.66      0.68       111

    accuracy                           0.65       594
   macro avg       0.66      0.63      0.64       594
weighted avg       0.65      0.65      0.64       594



#### Logistic Regression

In [69]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        max_iter=2000,
        multi_class="multinomial",
        solver="lbfgs",
        class_weight="balanced",
        n_jobs=-1
    ))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_proba = log_model.predict_proba(X_test)

auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(y_test, y_pred))

/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



===== Logistic Regression =====
Accuracy: 0.5589225589225589
Macro F1: 0.5564282700239017
Macro AUC: 0.8082353787341561
              precision    recall  f1-score   support

   Fertility       0.52      0.40      0.46       131
  Follicular       0.49      0.50      0.49       161
      Luteal       0.60      0.62      0.61       191
   Menstrual       0.62      0.72      0.67       111

    accuracy                           0.56       594
   macro avg       0.56      0.56      0.56       594
weighted avg       0.56      0.56      0.55       594



#### XGBoost

In [70]:
from sklearn.preprocessing import LabelEncoder

y_raw = df_backward[TARGET_COL]

le = LabelEncoder()
y = le.fit_transform(y_raw)

mask = ~pd.isna(y)
X = df_backward.drop(columns=[TARGET_COL])[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [71]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

# class weight（balanced）
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))

sample_weight = np.array([class_weight_dict[c] for c in y_train])

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(classes),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train, clf__sample_weight=sample_weight)

y_pred_int = xgb_model.predict(X_test)
y_pred = le.inverse_transform(y_pred_int)

y_proba = xgb_model.predict_proba(X_test)

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
auc_macro = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

print("\n===== XGBoost =====")
print("Accuracy:", accuracy_score(le.inverse_transform(y_test), y_pred))
print("Macro F1:", f1_score(le.inverse_transform(y_test), y_pred, average="macro"))
print("Macro AUC:", auc_macro)
print(classification_report(le.inverse_transform(y_test), y_pred))


===== XGBoost =====
Accuracy: 0.6262626262626263
Macro F1: 0.6230685310000335
Macro AUC: 0.8470803217574701
              precision    recall  f1-score   support

   Fertility       0.55      0.53      0.54       131
  Follicular       0.58      0.61      0.60       161
      Luteal       0.69      0.68      0.68       191
   Menstrual       0.67      0.67      0.67       111

    accuracy                           0.63       594
   macro avg       0.62      0.62      0.62       594
weighted avg       0.63      0.63      0.63       594

